# Weather Competition Jupyter Runner

Run these cells from top to bottom after uploading the project to JupyterLab. Start with the smoke test settings, then switch to full training after the environment and data paths are confirmed.

In [ ]:
# 1. Check runtime
import os
import sys
import torch

print(sys.version)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Check project and platform paths
from pathlib import Path

ROOT = Path('.').resolve()
print('ROOT:', ROOT)
print([p.name for p in ROOT.iterdir()])

!python scripts/check_platform_paths.py

In [ ]:
# 3. Set paths if the platform dataset is not under data/train and data/test
# os.environ['TRAIN_DIR'] = 'actual_train_dir'
# os.environ['TEST_DIR'] = 'actual_test_dir'
# os.environ['SAMPLE_SUBMISSION_PATH'] = 'actual_sample_submission.csv'

print('TRAIN_DIR =', os.environ.get('TRAIN_DIR', 'data/train'))
print('TEST_DIR =', os.environ.get('TEST_DIR', 'data/test'))
print('SAMPLE_SUBMISSION_PATH =', os.environ.get('SAMPLE_SUBMISSION_PATH', 'data/sample_submission.csv'))

In [ ]:
# 4. Check training data
from src.dataset import scan_image_folder

train_dir = os.environ.get('TRAIN_DIR', 'data/train')
df = scan_image_folder(train_dir)
print(df['label'].value_counts().sort_index())
print('total:', len(df))

In [ ]:
# 5. Lightweight smoke-test parameters for first JupyterLab run
os.environ['MODEL_NAME'] = 'efficientnet_b0'
os.environ['FALLBACK_MODEL_NAME'] = 'efficientnet_b0'
os.environ['PRETRAINED'] = 'false'
os.environ['IMG_SIZE'] = '224'
os.environ['BATCH_SIZE'] = '8'
os.environ['EPOCHS'] = '1'
os.environ['NUM_WORKERS'] = '0'

for key in ['MODEL_NAME', 'PRETRAINED', 'IMG_SIZE', 'BATCH_SIZE', 'EPOCHS', 'NUM_WORKERS']:
    print(key, os.environ[key])

In [ ]:
# 6. Run isolated synthetic smoke test
# This does not need official test images and writes generated files under tmp/smoke_test.
!python scripts/smoke_test.py --epochs 1 --batch-size 8 --model-name efficientnet_b0 --fallback-model-name efficientnet_b0

In [ ]:
# 7. Switch to full training settings after smoke test passes
# Adjust BATCH_SIZE for GPU memory.
os.environ['MODEL_NAME'] = 'convnext_tiny'
os.environ['FALLBACK_MODEL_NAME'] = 'resnet50'
os.environ['PRETRAINED'] = 'true'
os.environ['IMG_SIZE'] = '224'
os.environ['BATCH_SIZE'] = '16'
os.environ['EPOCHS'] = '30'
os.environ['NUM_WORKERS'] = '2'

for key in ['MODEL_NAME', 'PRETRAINED', 'IMG_SIZE', 'BATCH_SIZE', 'EPOCHS', 'NUM_WORKERS']:
    print(key, os.environ[key])

In [ ]:
# 8. Run full training
!python train.py

In [ ]:
# 9. Run inference after official test images are available
!python infer.py

In [ ]:
# 10. Check generated files and submission CSV
import pandas as pd

paths = [
    Path('results/best_model.pth'),
    Path('results/training_summary.json'),
    Path('logs/train_log.csv'),
    Path('outputs/submissions/submission.csv'),
    Path('outputs/confusion_matrix.png'),
]
for path in paths:
    print(path, path.exists())

sub_path = Path('outputs/submissions/submission.csv')
if sub_path.exists():
    sub = pd.read_csv(sub_path)
    print(sub.head())
    print(sub.shape)
    print(sub.isnull().sum())